# 02 Cleaning

Use this notebook to clean the raw data, document your transformations, and export a processed file to `data/processed/`.

In [ ]:
import pandas as pd
import os

raw_data_path = '../data/raw/'
processed_data_path = '../data/processed/'

os.makedirs(processed_data_path, exist_ok=True)

# Load data
account_statuses = pd.read_csv(os.path.join(raw_data_path, 'account_statuses.csv'))
account_types = pd.read_csv(os.path.join(raw_data_path, 'account_types.csv'))
accounts = pd.read_csv(os.path.join(raw_data_path, 'accounts.csv'))
addresses = pd.read_csv(os.path.join(raw_data_path, 'addresses.csv'))
branches = pd.read_csv(os.path.join(raw_data_path, 'branches.csv'))
customer_types = pd.read_csv(os.path.join(raw_data_path, 'customer_types.csv'))
customers = pd.read_csv(os.path.join(raw_data_path, 'customers.csv'))
loan_statuses = pd.read_csv(os.path.join(raw_data_path, 'loan_statuses.csv'))
loans = pd.read_csv(os.path.join(raw_data_path, 'loans.csv'))
transaction_types = pd.read_csv(os.path.join(raw_data_path, 'transaction_types.csv'))
transactions = pd.read_csv(os.path.join(raw_data_path, 'transactions.csv'))

## 1. Create dim_customers_accounts
We join customers with their addresses, types, and then link them to their accounts.

In [ ]:
# Join customers with types and addresses
customers_full = customers.merge(customer_types, on='CustomerTypeID', how='left', suffixes=('', '_Type'))
customers_full = customers_full.merge(addresses, on='AddressID', how='left')

# Join accounts with types and statuses
accounts_full = accounts.merge(account_types, on='AccountTypeID', how='left', suffixes=('', '_Type'))
accounts_full = accounts_full.merge(account_statuses, on='AccountStatusID', how='left', suffixes=('', '_Status'))

# Consolidate Customer and Account information
dim_customers_accounts = accounts_full.merge(customers_full, on='CustomerID', how='left', suffixes=('_Account', '_Customer'))

print(f"dim_customers_accounts shape: {dim_customers_accounts.shape}")
dim_customers_accounts.head()

## 2. Create fact_transactions

In [ ]:
# Join branches with their addresses
branches_full = branches.merge(addresses, on='AddressID', how='left', suffixes=('', '_Branch'))

# Join transactions with types and branches
fact_transactions = transactions.merge(transaction_types, on='TransactionTypeID', how='left', suffixes=('', '_Type'))
fact_transactions = fact_transactions.merge(branches_full, on='BranchID', how='left', suffixes=('', '_Branch'))

print(f"fact_transactions shape: {fact_transactions.shape}")

## 3. Create fact_loans

In [ ]:
# Join loans with statuses
fact_loans = loans.merge(loan_statuses, on='LoanStatusID', how='left', suffixes=('', '_Status'))
print(f"fact_loans shape: {fact_loans.shape}")

## Save the consolidated tables

In [ ]:
dim_customers_accounts.to_csv(os.path.join(processed_data_path, 'dim_customers_accounts.csv'), index=False)
fact_transactions.to_csv(os.path.join(processed_data_path, 'fact_transactions.csv'), index=False)
fact_loans.to_csv(os.path.join(processed_data_path, 'fact_loans.csv'), index=False)
print("Consolidated tables saved to data/processed/")